# 📊 Customer Churn Prediction — EDA & Model Experiments

Notebook lưu vết toàn bộ quá trình khám phá dữ liệu (EDA), tiền xử lý, huấn luyện, so sánh và trực quan hóa 4 mô hình Machine Learning trên tập dữ liệu **Telco Customer Churn**.

---

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath(".."))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from xgboost import XGBClassifier

from src.data_loader import (
    CATEGORICAL_FEATURES,
    NUMERICAL_FEATURES,
    clean_data,
    get_train_test_data,
    load_raw_data,
)
from src.pipeline import build_full_pipeline, build_preprocessor

IMAGES_DIR = os.path.join("..", "docs", "images")
os.makedirs(IMAGES_DIR, exist_ok=True)

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)

## Phần 1: EDA & Khám phá dữ liệu

### 1.1 Đọc dữ liệu thô và kiểm tra kiểu dữ liệu, giá trị thiếu

In [ ]:
raw_df = load_raw_data()
print(f"Kích thước dữ liệu: {raw_df.shape[0]} dòng, {raw_df.shape[1]} cột")
raw_df.head()

In [ ]:
raw_df.info()

In [ ]:
explicit_missing = raw_df.isnull().sum()
blank_total_charges = (raw_df["TotalCharges"].astype(str).str.strip() == "").sum()

print("Giá trị NaN tường minh theo từng cột:")
print(explicit_missing[explicit_missing > 0] if explicit_missing.sum() > 0 else "Không có NaN tường minh")
print(f"\nSố dòng có TotalCharges là chuỗi rỗng: {blank_total_charges}")
raw_df[raw_df["TotalCharges"].astype(str).str.strip() == ""][["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]]

### 1.2 Phân phối biến mục tiêu (`Churn`) — Kiểm tra mất cân bằng lớp

In [ ]:
churn_counts = raw_df["Churn"].value_counts()
churn_ratios = raw_df["Churn"].value_counts(normalize=True) * 100

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(churn_counts.index, churn_counts.values, color=["#4C72B0", "#DD8452"])
for bar, pct in zip(bars, churn_ratios.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 80,
        f"{int(bar.get_height())} ({pct:.1f}%)",
        ha="center",
        fontweight="bold",
    )
ax.set_title("Phân phối biến mục tiêu Churn")
ax.set_ylabel("Số lượng khách hàng")
ax.set_ylim(0, churn_counts.max() * 1.15)
plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, "01_churn_distribution.png"), dpi=150, bbox_inches="tight")
plt.show()

**Phân tích biểu đồ:** Tập dữ liệu có sự mất cân bằng lớp rõ rệt: **5,174** khách hàng ở lại (`No`, **73.5%**) so với **1,869** khách hàng rời bỏ (`Yes`, **26.5%**), tỷ lệ ~2.77:1. Vì vậy không thể dùng Accuracy làm tiêu chí chính mà phải kết hợp Stratified Split, trọng số cân bằng lớp (`scale_pos_weight` / `class_weight="balanced"`) và đánh giá theo **ROC-AUC** & **Recall**.

### 1.3 Mối quan hệ giữa `tenure`, `MonthlyCharges`, `Contract` với `Churn`

In [ ]:
cleaned_df = clean_data(raw_df)
cleaned_df["ChurnLabel"] = cleaned_df["Churn"].map({0: "No", 1: "Yes"})

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(data=cleaned_df, x="tenure", hue="ChurnLabel", kde=True, bins=30, ax=axes[0])
axes[0].set_title("Phân phối Tenure theo Churn")

sns.boxplot(data=cleaned_df, x="ChurnLabel", y="MonthlyCharges", ax=axes[1])
axes[1].set_title("MonthlyCharges theo Churn")

contract_churn = (
    cleaned_df.groupby("Contract")["Churn"]
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)
sns.barplot(x=contract_churn.index, y=contract_churn.values, ax=axes[2], color="#DD8452")
axes[2].set_title("Tỷ lệ Churn (%) theo Loại hợp đồng")
axes[2].set_ylabel("Tỷ lệ Churn (%)")
for idx, val in enumerate(contract_churn.values):
    axes[2].text(idx, val + 1, f"{val:.1f}%", ha="center", fontweight="bold")
axes[2].set_ylim(0, contract_churn.max() * 1.18)

plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, "02_eda_features_vs_churn.png"), dpi=150, bbox_inches="tight")
plt.show()

**Phân tích biểu đồ:**
- **`tenure`:** Nhóm Churn có trung vị thời gian gắn bó chỉ **10 tháng** (so với **38 tháng** ở nhóm ở lại) — khách hàng mới trong năm đầu có rủi ro rời bỏ cao nhất.
- **`MonthlyCharges`:** Trung vị cước tháng của nhóm Churn là **79.65 USD**, cao hơn đáng kể so với nhóm ở lại (**64.43 USD**).
- **`Contract`:** Hợp đồng `Month-to-month` có tỷ lệ rời bỏ tới **42.7%**, cao hơn nhiều so với `One year` (**11.3%**) và `Two year` (**2.8%**).

--- 
## Phần 2: Data Preprocessing

Chia tập dữ liệu Stratified 80/20 và minh họa quá trình chuẩn hóa (`StandardScaler`) kết hợp mã hóa (`OneHotEncoder`) thông qua `ColumnTransformer`.

In [ ]:
X_train, X_test, y_train, y_test = get_train_test_data()
print(f"X_train: {X_train.shape}, y_train Churn rate: {y_train.mean():.4f}")
print(f"X_test:  {X_test.shape}, y_test Churn rate:  {y_test.mean():.4f}")

preprocessor = build_preprocessor()
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()
print(f"Số chiều sau One-Hot + Scaling: {X_train_transformed.shape[1]} features")

In [ ]:
scaled_df = pd.DataFrame(
    X_train_transformed[:, : len(NUMERICAL_FEATURES)],
    columns=NUMERICAL_FEATURES,
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.kdeplot(data=X_train[NUMERICAL_FEATURES], ax=axes[0])
axes[0].set_title("Trước StandardScaler (thang đo chênh lệch lớn)")

sns.kdeplot(data=scaled_df, ax=axes[1])
axes[1].set_title("Sau StandardScaler (mean = 0, std = 1)")

plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, "03_standard_scaler_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()

**Phân tích biểu đồ:** Trước khi scale, `TotalCharges` kéo trục hoành tới ~`10,000` làm `tenure` và `MonthlyCharges` bị ép thành 2 vạch sát `0`. Sau `StandardScaler`, cả 3 biến được đưa về cùng thang đo quanh `[-2, +3]` với tâm `0` mà vẫn giữ nguyên hình dạng phân phối (2 đỉnh ở `tenure`, `MonthlyCharges` và lệch phải ở `TotalCharges`).

--- 
## Phần 3: Huấn luyện & So sánh mô hình

Thực nghiệm 4 thuật toán: **Logistic Regression**, **Random Forest**, **XGBoost**, **LightGBM** với xử lý mất cân bằng lớp (`class_weight="balanced"` / `scale_pos_weight`).

In [ ]:
scale_pos_weight = (len(y_train) - y_train.sum()) / y_train.sum()

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000, class_weight="balanced", random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=150,
        max_depth=8,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    ),
    "XGBoost": XGBClassifier(
        n_estimators=150,
        max_depth=4,
        learning_rate=0.05,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        eval_metric="logloss",
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=150,
        max_depth=5,
        learning_rate=0.05,
        class_weight="balanced",
        random_state=42,
        verbose=-1,
    ),
}

results = []
trained_pipelines = {}
probabilities = {}
predictions = {}

for name, estimator in models.items():
    pipeline = build_full_pipeline(estimator)
    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]

    trained_pipelines[name] = pipeline
    predictions[name] = y_pred
    probabilities[name] = y_prob

    results.append(
        {
            "Model": name,
            "ROC-AUC": round(roc_auc_score(y_test, y_prob), 4),
            "Recall": round(recall_score(y_test, y_pred), 4),
            "Precision": round(precision_score(y_test, y_pred), 4),
            "F1-Score": round(f1_score(y_test, y_pred), 4),
            "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        }
    )

results_df = pd.DataFrame(results).sort_values(by="ROC-AUC", ascending=False).reset_index(drop=True)
results_df

--- 
## Phần 4: Đánh giá & Trực quan hóa kết quả

### 4.1 Đường cong ROC (ROC Curve) so sánh 4 mô hình

In [ ]:
plt.figure(figsize=(8, 6))
for name, y_prob in probabilities.items():
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc_val = roc_auc_score(y_test, y_prob)
    plt.plot(fpr, tpr, linewidth=2, label=f"{name} (AUC = {auc_val:.4f})")

plt.plot([0, 1], [0, 1], "k--", alpha=0.6, label="Random Classifier (AUC = 0.5000)")
plt.title("So sánh đường cong ROC giữa 4 mô hình")
plt.xlabel("False Positive Rate (FPR)")
plt.ylabel("True Positive Rate (Recall)")
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, "04_roc_curves_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()

**Phân tích biểu đồ:** Cả 4 mô hình đều đạt khả năng phân tách vượt trội so với đường đoán ngẫu nhiên (`AUC = 0.5000`). **XGBoost** đứng đầu với `ROC-AUC = 0.8435` (và `Recall = 0.8021`), theo sát bởi Logistic Regression (`0.8415`), Random Forest (`0.8414`) và LightGBM (`0.8414`).

### 4.2 Ma trận nhầm lẫn (Confusion Matrix) & Feature Importance của XGBoost

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_pipeline = trained_pipelines[best_model_name]
best_cm = confusion_matrix(y_test, predictions[best_model_name])

encoded_names = best_pipeline.named_steps["preprocessor"].get_feature_names_out()
importances = best_pipeline.named_steps["classifier"].feature_importances_
importance_df = (
    pd.DataFrame({"Feature": encoded_names, "Importance": importances})
    .sort_values(by="Importance", ascending=False)
    .head(15)
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

disp = ConfusionMatrixDisplay(
    confusion_matrix=best_cm,
    display_labels=["Retain (0)", "Churn (1)"],
)
disp.plot(ax=axes[0], cmap="Blues", colorbar=False)
axes[0].set_title(f"Confusion Matrix — {best_model_name}")
axes[0].grid(False)

sns.barplot(data=importance_df, x="Importance", y="Feature", ax=axes[1], color="#4C72B0")
axes[1].set_title(f"Top 15 Feature Importances — {best_model_name}")

plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, "05_confusion_matrix_feature_importance.png"), dpi=150, bbox_inches="tight")
plt.show()

**Phân tích biểu đồ:**
- **Confusion Matrix:** Trên 1,409 mẫu test, XGBoost dự đoán đúng **300 / 374** khách hàng Churn (**Recall = 80.21%**), chỉ bỏ sót 74 mẫu (`FN = 74`).
- **Feature Importance:** `cat__Contract_Month-to-month` chiếm tỷ trọng lớn nhất (~**52.5%**), tiếp theo là `cat__InternetService_Fiber optic` (~**9.9%**), `cat__OnlineSecurity_No` (~**4.6%**) và `cat__TechSupport_No` (~**3.7%**).